# 🚀 AnyProjector v0.9.4 — Phase 3: End-to-End Fine-Tuning (Speech-to-Intent)

Train Phase 3: Biến mô hình thành một trợ lý thực thi Tool-Calling trực tiếp từ Audio!

**Kiến trúc:**
- **Whisper (Audio Encoder)**: Mở khóa 2 layers cuối cùng để học biểu diễn audio chuyên sâu hơn cho tool-calling.
- **Projector (Q-Former)**: Freeze hoàn toàn (đã được align tốt ở Phase 2).
- **LLM (Qwen2.5-1.5B)**: Apply LoRA (r=16, target q_proj, v_proj) để học format tool-calling sinh ra.

**Dataset:** Dataset `speech-massive-vie-tool-calling` có chứa cột `audio`, `instruction` (bối cảnh + intent) và `output` (câu lệnh JSON tool call sinh ra).


In [ ]:
!pip install -q transformers datasets torch accelerate hf_transfer huggingface_hub matplotlib peft
import os
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'
print('✅ Dependencies installed')

In [ ]:
import torch
print(f'CUDA: {torch.cuda.is_available()} | GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB')

In [ ]:
from huggingface_hub import login
login() # Nhập token HuggingFace của bạn để lấy dataset

In [ ]:
# ============================================
# 🔧 Phase 3 Config
# ============================================
ENCODER_ID    = "openai/whisper-medium"
LLM_ID        = "Qwen/Qwen2.5-1.5B-Instruct"

# THAY THẾ BẰNG DATASET CỦA BẠN!
DATASET_ID    = "YOUR_USERNAME/speech-massive-vie-tool-calling"

# --- Training Params ---
NUM_EPOCHS  = 10
BATCH_SIZE  = 4      # T4 VRAM giới hạn ở 4-8
LR          = 5e-5   # LR nhỏ cho End-to-End tuning
GRAD_ACCUM  = 4      # Effective batch size = 16
SAVE_DIR    = "checkpoints/phase3/end2end_toolcalling"

# --- Phase 3 Specifics ---
PROJECTOR_WEIGHTS = "checkpoints/phase2/v094_vietspeech/projector_best.pt" # UPDATE ĐƯỜNG DẪN TỚI PROJECTOR CỦA PHASE 2!

UNFREEZE_ENCODER_LAYERS = 2  # Unfreeze 2 layers cuối của Whisper
LORA_ENABLED = True          # Bật LoRA cho LLM
LORA_RANK    = 16
LORA_ALPHA   = 32

NUM_QUERIES    = 64
QFORMER_DIM    = 768
QFORMER_LAYERS = 4
QFORMER_HEADS  = 16

In [ ]:
from datasets import load_dataset
import gc

print(f"Downloading dataset {DATASET_ID}...")
ds = load_dataset(DATASET_ID, split="train")
print(f"✅ Loaded {len(ds)} samples!")
print("Columns:", ds.column_names)

In [ ]:
import torch
import torch.nn as nn

# Copy nguyên class QFormerLayer & AnyProjector từ Phase 2
class QFormerLayer(nn.Module):
    def __init__(self, qformer_dim: int, encoder_dim: int, num_heads: int = 8, ffn_ratio: int = 4):
        super().__init__()
        self.self_attn = nn.MultiheadAttention(embed_dim=qformer_dim, num_heads=num_heads, batch_first=True)
        self.self_attn_norm = nn.LayerNorm(qformer_dim)
        self.cross_attn = nn.MultiheadAttention(embed_dim=qformer_dim, num_heads=num_heads, kdim=encoder_dim, vdim=encoder_dim, batch_first=True)
        self.cross_attn_norm = nn.LayerNorm(qformer_dim)
        ffn_hidden = qformer_dim * ffn_ratio
        self.ffn = nn.Sequential(nn.Linear(qformer_dim, ffn_hidden), nn.GELU(), nn.Linear(ffn_hidden, qformer_dim))
        self.ffn_norm = nn.LayerNorm(qformer_dim)

    def forward(self, queries: torch.Tensor, encoder_out: torch.Tensor, encoder_mask: torch.Tensor | None = None) -> torch.Tensor:
        q = self.self_attn_norm(queries)
        q, _ = self.self_attn(q, q, q)
        queries = queries + q
        q = self.cross_attn_norm(queries)
        q, _ = self.cross_attn(query=q, key=encoder_out, value=encoder_out, key_padding_mask=encoder_mask)
        queries = queries + q
        queries = queries + self.ffn(self.ffn_norm(queries))
        return queries

class AnyProjector(nn.Module):
    def __init__(self, encoder_dim: int, llm_dim: int, num_queries: int = 64, qformer_dim: int = 768, num_layers: int = 2, num_heads: int = 8):
        super().__init__()
        self.encoder_dim = encoder_dim
        self.llm_dim = llm_dim
        self.num_queries = num_queries
        self.qformer_dim = qformer_dim
        self.pre_proj = nn.Sequential(nn.Linear(encoder_dim, encoder_dim), nn.GELU(), nn.LayerNorm(encoder_dim))
        self.query_tokens = nn.Parameter(torch.randn(1, num_queries, qformer_dim) * 0.02)
        self.layers = nn.ModuleList([QFormerLayer(qformer_dim, encoder_dim, num_heads) for _ in range(num_layers)])
        self.output_norm = nn.LayerNorm(qformer_dim)
        self.output_proj = nn.Linear(qformer_dim, llm_dim)

    def forward(self, encoder_output: torch.Tensor, encoder_mask: torch.Tensor | None = None) -> torch.Tensor:
        batch_size = encoder_output.shape[0]
        encoder_output = self.pre_proj(encoder_output)
        queries = self.query_tokens.expand(batch_size, -1, -1)
        for layer in self.layers:
            queries = layer(queries, encoder_output, encoder_mask)
        queries = self.output_norm(queries)
        return self.output_proj(queries)

In [ ]:
from torch.utils.data import Dataset, DataLoader
import numpy as np

class Phase3Dataset(Dataset):
    def __init__(self, hf_ds, max_samples=480000): # max 30s * 16000
        self.ds = hf_ds
        self.max_samples = max_samples
    
    def __len__(self):
        return len(self.ds)
    
    def __getitem__(self, idx):
        row = self.ds[idx]
        # Audio array
        waveform = torch.from_numpy(row["audio"]["array"].astype(np.float32))
        if waveform.shape[0] > self.max_samples:
            waveform = waveform[:self.max_samples]
            
        return {
            "waveform": waveform,
            "instruction": row["instruction"],
            "output": row["output"],
        }

def collate_fn(batch):
    waveforms = [b["waveform"] for b in batch]
    waveforms_padded = nn.utils.rnn.pad_sequence(waveforms, batch_first=True, padding_value=0.0)
    return {
        "waveforms": waveforms_padded,
        "instructions": [b["instruction"] for b in batch],
        "outputs": [b["output"] for b in batch]
    }

train_ds = Phase3Dataset(ds)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn, num_workers=2)

In [ ]:
from transformers import WhisperProcessor, WhisperModel, AutoModelForCausalLM, AutoTokenizer, AutoConfig
from peft import LoraConfig, get_peft_model
import math

device = "cuda" if torch.cuda.is_available() else "cpu"

# 1. Load Whisper Encoder
print("Loading Whisper...")
processor = WhisperProcessor.from_pretrained(ENCODER_ID)
encoder = WhisperModel.from_pretrained(ENCODER_ID).encoder.to(device)
encoder.eval()

# Unfreeze top layers
for p in encoder.parameters():
    p.requires_grad = False

if UNFREEZE_ENCODER_LAYERS > 0:
    for layer in encoder.layers[-UNFREEZE_ENCODER_LAYERS:]:
        for p in layer.parameters():
            p.requires_grad = True
    print(f"✅ Unfrozen last {UNFREEZE_ENCODER_LAYERS} layers of Whisper")

# 2. Load LLM (Qwen) + LoRA
print("Loading Qwen2.5-1.5B...")
tokenizer = AutoTokenizer.from_pretrained(LLM_ID)
if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token

llm = AutoModelForCausalLM.from_pretrained(LLM_ID, device_map="auto", torch_dtype=torch.bfloat16)
for p in llm.parameters(): p.requires_grad = False

lora_config = LoraConfig(r=LORA_RANK, lora_alpha=LORA_ALPHA, target_modules=["q_proj", "v_proj", "k_proj", "o_proj"], bias="none", task_type="CAUSAL_LM")
llm = get_peft_model(llm, lora_config)
print(f"✅ LoRA applied to Qwen. Trainable params: {sum(p.numel() for p in llm.parameters() if p.requires_grad):,}")

# 3. Load Projector & Freeze it
print("Loading Projector...")
llm_dim = AutoConfig.from_pretrained(LLM_ID).hidden_size
projector = AnyProjector(encoder_dim=encoder.config.d_model, llm_dim=llm_dim, num_queries=NUM_QUERIES, qformer_dim=QFORMER_DIM, num_layers=QFORMER_LAYERS, num_heads=QFORMER_HEADS)

if os.path.exists(PROJECTOR_WEIGHTS):
    projector.load_state_dict(torch.load(PROJECTOR_WEIGHTS, map_location="cpu"))
    print(f"✅ Loaded pre-trained Phase 2 weights from {PROJECTOR_WEIGHTS}")
else:
    print(f"⚠️ WARNING: Projector weights {PROJECTOR_WEIGHTS} NOT FOUND. Initializing from scratch!")

projector.to(device)
for p in projector.parameters(): 
    p.requires_grad = False # BẮT BUỘC FREEZE TRONG PHASE 3
projector.eval()

# 4. Optimizer
trainable_params = [p for p in encoder.parameters() if p.requires_grad] + [p for p in llm.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(trainable_params, lr=LR, weight_decay=0.01)

In [ ]:
# @title 🚀 Training Loop End-To-End
import time

embed_layer = llm.get_base_model().get_input_embeddings()
llm_dtype = next(llm.parameters()).dtype

def forward_step(batch):
    waveforms = batch["waveforms"]
    instructions = batch["instructions"]
    targets = batch["outputs"]
    batch_size = len(waveforms)
    
    # 1. AUDIO → WHISPER → PROJECTOR
    audio_inputs = processor([w.numpy() for w in waveforms], sampling_rate=16000, return_tensors="pt", padding="max_length")
    input_features = audio_inputs.input_features.to(device)
    
    with torch.set_grad_enabled(UNFREEZE_ENCODER_LAYERS > 0):
        encoder_output = encoder(input_features).last_hidden_state
        
    with torch.no_grad(): # Projector is completely frozen
        audio_embeds = projector(encoder_output) # (B, 64, llm_dim)
        
    # 2. CONSTRUCT CHAT TEMPLATE EMBEDDINGS
    # Prefix: System prompt + User tag
    prefix_texts = [f"<|im_start|>system\n{inst}<|im_end|>\n<|im_start|>user\n" for inst in instructions]
    tokenizer.padding_side = "left" # Padding bên trái để audio align thẳng hàng
    prefix_tokens = tokenizer(prefix_texts, return_tensors="pt", padding=True, add_special_tokens=False).to(device)
    with torch.no_grad(): prefix_embeds = embed_layer(prefix_tokens.input_ids)
    prefix_mask = prefix_tokens.attention_mask
    
    # Audio Embeds mask
    audio_mask = torch.ones((batch_size, NUM_QUERIES), dtype=torch.long, device=device)
    
    # Suffix: Close User tag + Assistant tag
    suffix_text = "<|im_end|>\n<|im_start|>assistant\n"
    tokenizer.padding_side = "right" # Từ đây padding bên phải cho target
    suffix_tokens = tokenizer([suffix_text] * batch_size, return_tensors="pt", padding=True, add_special_tokens=False).to(device)
    with torch.no_grad(): suffix_embeds = embed_layer(suffix_tokens.input_ids)
    suffix_mask = suffix_tokens.attention_mask
    
    # Target (Outputs)
    target_texts = [f"{t}<|im_end|>" for t in targets]
    target_tokens = tokenizer(target_texts, return_tensors="pt", padding=True, add_special_tokens=False, truncation=True, max_length=128).to(device)
    with torch.no_grad(): target_embeds = embed_layer(target_tokens.input_ids)
    target_mask = target_tokens.attention_mask
    
    # 3. GỌP TẤT CẢ VÀ TẠO LABELS
    full_embeds = torch.cat([prefix_embeds, audio_embeds, suffix_embeds, target_embeds], dim=1).to(llm_dtype)
    full_mask = torch.cat([prefix_mask, audio_mask, suffix_mask, target_mask], dim=1)
    
    # Labels: -100 cho tất cả ngoại trừ target_ids
    ignore_prefix = torch.full_like(prefix_tokens.input_ids, -100)
    ignore_audio = torch.full_like(audio_mask, -100)
    ignore_suffix = torch.full_like(suffix_tokens.input_ids, -100)
    
    target_labels = target_tokens.input_ids.clone()
    target_labels[target_mask == 0] = -100 # Bỏ qua tính loss cho các token padding của target
    
    labels = torch.cat([ignore_prefix, ignore_audio, ignore_suffix, target_labels], dim=1)
    
    # 4. CHẠY LLM
    outputs = llm(inputs_embeds=full_embeds, attention_mask=full_mask, labels=labels)
    return outputs.loss


# MAIN LOOP
print("🔥 Bắt đầu Training End-to-End!")
global_step = 0
llm.train()
if UNFREEZE_ENCODER_LAYERS > 0: encoder.train()
    
for epoch in range(NUM_EPOCHS):
    optimizer.zero_grad()
    for step, batch in enumerate(train_loader):
        loss = forward_step(batch)
        loss = loss / GRAD_ACCUM
        loss.backward()
        
        if (step + 1) % GRAD_ACCUM == 0:
            optimizer.step()
            optimizer.zero_grad()
            global_step += 1
            
            if global_step % 10 == 0:
                print(f"Epoch {epoch} | Step {global_step} | Loss: {loss.item() * GRAD_ACCUM:.4f}")
        
    # Save sau mỗi epoch
    os.makedirs(f"{SAVE_DIR}/epoch_{epoch}", exist_ok=True)
    llm.save_pretrained(f"{SAVE_DIR}/epoch_{epoch}/llm_lora")
    torch.save(encoder.state_dict(), f"{SAVE_DIR}/epoch_{epoch}/whisper_encoder.pt")
    print(f"💾 Đã lưu Epoch {epoch}")
    
print("🎉 Train xong!")